 **Dataset**
--
    Data Science EDA
    
### **Data Description**
This dataset contains medical insurance cost information for 1338 individuals. It includes demographic and health-related variables such as age, sex, BMI, number of children, smoking status, and residential region in the US. The target variable is charges, which represents the medical insurance cost billed to the individual.

### **Features**
    Building Type: Categorical feature representing the type of building.
    Sex: Male of Female
    BMI: Body mass index
    Children: Number of Children 
    Smoker: TRUE or FALSE
    Region: Region where the payer lives
    Charges: Medical insurance cost billed to the beneficiary

## **Import Libraries**

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import warnings
warnings.filterwarnings("ignore")

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# **EDA**

## **Data Exploration**
    During this step we can explore the shape of the data we are working with. It's usual to check and deal with the number of columns and rows, missing values, outliers and other inconsistencies that might appear.
--

In [ ]:
dataframe = pd.read_csv('/kaggle/input/medical-insurance-cost-dataset/insurance.csv')
dataframe

In [ ]:
features = {
    'age': 'Age of primary beneficiary',
    'sex': 'Gender of beneficiary (1: Male, 2: Female)',
    'bmi': 'Body Mass Index, a measure of body fat based on height and weight',
    'children': 'Number of children covered by health insurance',
    'smoker': 'Smoking status of the beneficiary',
    'region': 'Residential region in the US (northeast, northwest, southeast, southwest)',
    'charges': 'Medical insurance cost billed to the beneficiary (float)'   
}

In [ ]:
data_assessment = pd.DataFrame({
    "Column": dataframe.columns,
    "Data Type": [dataframe[col].dtype for col in dataframe.columns],
    "Missing Values": [f"{dataframe[col].isna().mean() * 100:.1f}%" for col in dataframe.columns],
    "Unique Values": [dataframe[col].nunique() for col in dataframe.columns],
    "Example Value": [dataframe[col].dropna().iloc[0] if dataframe[col].notna().any() else None for col in dataframe.columns]
})
data_assessment

In [ ]:
duplicate_rows_data = dataframe[dataframe.duplicated()]
print("number of duplicate rows: ", duplicate_rows_data.shape)

dataframe.drop_duplicates(inplace=True)

In [ ]:
dataframe

In [ ]:
dataframe['sex'] = dataframe['sex'].map({
    "male": 1,
    "female": 2
})

dataframe['smoker'] = dataframe['smoker'].map({
    "yes": 1,
    "no": 0
})

dataframe['charges'] = dataframe['charges'].round(2)

## **Statistical Analysis**
    This step seeks to understand the interaction and correlation among the features. Here it's possible to check multicolinearity, plot visulizations and highlight key points.
--

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

##### **Distribution of features**

In [ ]:
dataframe[['age','bmi','charges']].describe().round(2)

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(12, 4))

ax[0] = sns.histplot(data=dataframe, x= 'age', ax=ax[0],)
ax[1] = sns.histplot(data=dataframe, x= 'bmi', ax=ax[1])
ax[2] = sns.histplot(data=dataframe, x= 'charges', ax=ax[2])


sns.despine(left=False, right=True, top=True, bottom=False)
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(12, 4))

ax[0] = sns.boxplot(data=dataframe, y= 'age', ax=ax[0],)
ax[1] = sns.boxplot(data=dataframe, y= 'bmi', ax=ax[1])
ax[2] = sns.boxplot(data=dataframe, y= 'charges', ax=ax[2])


sns.despine(left=False, right=True, top=True, bottom=False)
plt.tight_layout()

##### **Missing values**

In [ ]:
dataframe.isna().mean() # Evaluating the % of missing values in every column

##### **Checking Outliers**

In [ ]:
# Calculate Q1, Q3, and IQR
Q1 = dataframe['charges'].quantile(0.25)
Q3 = dataframe['charges'].quantile(0.75)
IQR = Q3 - Q1

# Define bounds
lower_bound = Q1 - (1.5 * IQR)
upper_bound = Q3 + (1.5 * IQR)

# Detect outliers
outliers = dataframe[(dataframe['charges'] < lower_bound) | (dataframe['charges'] > upper_bound)]

outliers_mask = (dataframe['charges'] < lower_bound) | (dataframe['charges'] > upper_bound)
clean_dataframe = dataframe[~outliers_mask]

In [ ]:
clean_dataframe

In [ ]:
print(f'The Outliers represent just {(outliers.shape[0]/dataframe.shape[0]):.2%}, therefore they will not be deleted for now as they might represent real data.')

##### **Exploring features**

In [ ]:
# Checking the AVG BMI per gender

sex_bmi = dataframe.groupby('sex')['bmi'].mean()
print(sex_bmi)
print(features['sex'])

In [ ]:
# Checking if people with more children tend to have a bigger or lower BMI

children_bmi = dataframe.groupby('children')['bmi'].mean()
print(children_bmi)
print(features['children'])

The AVG charge according to the number of children shows minimal difference.

##### **Correlation of features**

In [ ]:
corr = dataframe.select_dtypes(include='number').corr()
mask = np.triu(np.ones_like(corr, dtype=bool)) # Mask to remove the upper side and facilitate visualization

plt.figure(figsize=(12,4))
sns.heatmap(corr, mask=mask, annot=True, cmap='coolwarm', center=0)

sns.despine(left=False, right=True, top=True, bottom=False)
plt.title("Correlation Analysis", fontsize=9)
plt.tight_layout()

In [ ]:
# Checking the movement of the variables

fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(12, 4))

# Plotting visualization

ax[0] = sns.scatterplot(data= dataframe, x=dataframe['age'], y=dataframe['charges'], ax=ax[0])
ax[1] = sns.scatterplot(data= dataframe, x=dataframe['bmi'], y=dataframe['charges'], ax=ax[1])
ax[2] = sns.scatterplot(data= dataframe, x=dataframe['smoker'], y=dataframe['charges'], ax=ax[2])

sns.despine(left=False, right=True, top=True, bottom=False)
plt.tight_layout()

In [ ]:
# Checking the movement of the variables

fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(12, 4))

# Plotting visualization
ax[0] = sns.scatterplot(data= dataframe, x=dataframe['bmi'], y=dataframe['charges'], hue='smoker', ax=ax[0])
ax[0].legend(title='Smoker', loc='upper right', ncol=1,labels=['Yes','No'], frameon=False, fontsize=9.5)

ax[1] = sns.scatterplot(data= dataframe, x=dataframe['charges'], y=dataframe['age'], hue='smoker', ax=ax[1])
ax[1].legend(title='Smoker', loc='upper right', ncol=1,labels=['Yes','No'], frameon=False, fontsize=9.5)


sns.despine(left=False, right=True, top=True, bottom=False)
plt.tight_layout()

According to the dataset, people who smoke usually have higher charges when paying for the insurance.

In [ ]:
region_group = dataframe.groupby('region')['charges'].median().round(2)
region_group

The median charge per region shows minimal difference.

## **Insights**
--

### **Statistics**:
* Based on the data provided in this dataset, it's possible to see that people who smoke usually have higher charges when paying for the insurance.
* There's no disparity in the charges, no matter the age, gender, number of children or BMI if the person isn't smoker.

# **Predctive Analysis**

## **Data Preparation**
    To prepare the data to build and test the models. 
--

In [ ]:
df_encoded = pd.get_dummies(dataframe, columns=['region'])
df_encoded[['region_northeast','region_northwest','region_southeast','region_southwest']] = df_encoded[['region_northeast','region_northwest','region_southeast','region_southwest']].astype(int)
df_encoded

In [ ]:
# Correlation of each feature against the target.

corr_encoded = df_encoded.corr()
target_corr = corr_encoded[['charges']].sort_values(by='charges', ascending=False)

plt.figure(figsize=(5, len(target_corr) * 0.5))
sns.heatmap(target_corr, annot=True, cmap='coolwarm', center=0)

plt.title("Target x Features")
plt.tight_layout()

In [ ]:
# Checking for Multicolinearity
# No multicolineatiry detected. We have a small correlation among the region features but nothing that stands out.

df_filter = corr_encoded[((corr_encoded >= 0.30) | (corr_encoded <= -0.30)) & (corr_encoded != 1.000)]
df_filter = df_filter.drop('charges', axis=1)
df_filter = df_filter.drop('charges', axis=0)
df_filter

There's no multicolineatiry present in this data set, but since the region, children and sex features doesn't show a significant correlation to the target feature, they will be removed from the dataset in the next steps.

In [ ]:
model_data = df_encoded.drop(['sex','children','region_northeast','region_northwest','region_southeast','region_southwest'], axis=1)

In [ ]:
# Function to run metrics

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    max_error
)

def regression_metrics(y_true, y_pred, round_digits=4):
    print("Evaluation Metrics:\n")
    
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    max_err = max_error(y_true, y_pred)

    print(f"R² Score: {r2:.{round_digits}f}")
    print(f"MAE: {mae:.{round_digits}f}")
    print(f"MSE: {mse:.{round_digits}f}")
    print(f"RMSE: {rmse:.{round_digits}f}")
    print(f"MAPE(%): {mape * 100:.{round_digits}f}%")
    print(f"Max Error: {max_err:.{round_digits}f}")

##### **Spliting and Scaling features**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Splitting and adjusting the dataset for training purposes

X = model_data.drop('charges', axis=1)
y = model_data['charges']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, shuffle=True)

# Initialize the StandardScaler
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled Data (Standardization):\n", X_scaled)

## **Testing Models**
    To test and validade base models, run metrics and confidence interval to evaluate model performance
--

##### **Linear Regression**

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan, acorr_ljungbox
from scipy import stats

# Add a constant (intercept) to the model
X = sm.add_constant(X_scaled)

# Fit the OLS model
model = sm.OLS(y, X)
results = model.fit()

# Print results
print(results.summary())

In [ ]:
residuals = results.resid
fitted = results.fittedvalues

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(12, 4))

plt.scatter(fitted, residuals)
plt.axhline(y=0, color='red', linestyle='--')
plt.title("Residuals vs Fitted")
plt.xlabel("Fitted values")
plt.ylabel("Residuals")

sns.despine(left=False, right=True, top=True, bottom=False)
plt.tight_layout()

In [ ]:
residuals.mean().round()

In [ ]:
# Checking if outliers are the main driver of the residuals pattern.

import numpy as np

std_resid = residuals / residuals.std()
outliers = np.abs(std_resid) > 3
outliers.sum()

percentage = outliers.sum() / df_encoded.shape[0]
percentage.round(2)*100

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(12, 4))

plt.hist(residuals, bins=20, edgecolor='black')
plt.title("Histogram of Residuals")
plt.show()

sns.despine(left=False, right=True, top=True, bottom=False)
plt.tight_layout()

The distribution of the residuals (**difference between predicted and actual values**) is not symmetric around 0 when it comes to big values. This means that the model is adjusting well to small values but not the big ones, showing Bias to some degree.

In [ ]:
# Linear Regression model
from sklearn.linear_model import LinearRegression

regression_model = LinearRegression()
regression_model.fit(X_train_scaled, y_train)

# Make predictions
y_pred = regression_model.predict(X_test_scaled)
regression_metrics(y_test, y_pred, round_digits=2)

Despite a good R2, MAE is indicating that the model is missing predictions around 4,216 units on average. This is supported by the MSE AND RMSE. The MAPE also indicates that the model is missing predictions by 41%. Let's try transforning the Dependent variable to improve the metrics.

##### **Linear Regression using LOG standartization of the Independent feature**

In [ ]:
# Splitting and adjusting the dataset for training purposes

X = model_data.drop('charges', axis=1)
y = model_data['charges']
y = np.log(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, shuffle=True)

# Initialize the StandardScaler
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan, acorr_ljungbox
from scipy import stats

# Add a constant (intercept) to the model
X = sm.add_constant(X_scaled)

# Fit the OLS model
model = sm.OLS(y, X)
results = model.fit()

# Print results
print(results.summary())

In [ ]:
residuals = results.resid
fitted = results.fittedvalues

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(12, 4))

plt.scatter(fitted, residuals)
plt.axhline(y=0, color='red', linestyle='--')
plt.title("Residuals vs Fitted")
plt.xlabel("Fitted values")
plt.ylabel("Residuals")

sns.despine(left=False, right=True, top=True, bottom=False)
plt.tight_layout()

In [ ]:
residuals.mean().round()

In [ ]:
#Checking if outliers are the main driver of the residuals pattern.

import numpy as np

std_resid = residuals / residuals.std()
outliers = np.abs(std_resid) > 3
outliers.sum()

percentage = outliers.sum() / df_encoded.shape[0]
percentage.round(2)*100

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(12, 4))

plt.hist(residuals, bins=20, edgecolor='black')
plt.title("Histogram of Residuals")
plt.show()

sns.despine(left=False, right=True, top=True, bottom=False)
plt.tight_layout()

The distribution of the residuals (**difference between predicted and actual values**) is not symmetric around 0 still showing some drift. The model is now adjusting better to all values, but is not necessarily good. It's still biased.

In [ ]:
# Linear Regression model
from sklearn.linear_model import LinearRegression

regression_model = LinearRegression()
regression_model.fit(X_train_scaled, y_train)

# Make predictions
y_pred = regression_model.predict(X_test_scaled)
regression_metrics(y_test, y_pred, round_digits=2)

The Linear regression model using LOG transformation shows a better result than the usual model. Yet, the R² score could be improve and it's still showing some errors.

#### **Decision Tree Regression**

In [ ]:
from sklearn.tree import DecisionTreeRegressor

X = model_data.drop('charges', axis=1)
y = model_data['charges']
y = np.log(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, shuffle=True)

tree = DecisionTreeRegressor(max_depth=5,random_state=42)
tree.fit(X_train, y_train)

# Predict
y_pred = tree.predict(X_test)
regression_metrics(y_test, y_pred, round_digits=2)

The Decision Tree showed a better result overall metrics. This is the model that will be used to make the necessary predictions for this dataset.

In [ ]:
y_pred = np.exp(y_pred)
y_test = np.exp(y_test)

In [ ]:
results = pd.DataFrame({
    'Actual': y_test.round(),
    'Predicted': y_pred.round(),
}).reset_index(drop=True)

results.head()

In [ ]:
results['Pct_Error'] = ((results['Predicted'] - results['Actual']).abs() / results['Actual']).round(2) *100
results.head(10)

## **Insights**
--

### **Base Models**:
* The OLS models are not adjusting well to the dataframe used. Therefore the data needs to be cleaned even more for better models, otherwise they can be considered biased models.
* All base models could be adjusted to reach a better R² score and error results. The Decision Tree model shows the best result, yet the model is missing many predictions when it comes to big amounts.
* XGBOOST and Cross Validation could bring improvements to the base models if applied correctly.
* The idea of this notebook is to show how data that wansn't adjusted correctly can bting bias to prediction models.

# **Thank You for taking the time to view this Notebook**!
​ 
If you found this analysis useful and have any feedback or suggestion, don't hesitate to contact me! We are here to learn!